[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alextfkd/TorchCode/blob/master/solutions/04_layernorm_solution.ipynb)

# ✅ Solution: LayerNorm の実装

**Layer Normalization** をスクラッチで実装する。

$$\text{LayerNorm}(x) = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

ここで $\mu$ と $\sigma^2$ は **最後の次元** で計算する。


> 💡 **どこで使う**：Transformer の標準正規化。sample 内で正規化するので batch size に依存しない。

<details>
<summary>📖 もっと詳しく</summary>

BatchNorm と違って batch 統計を持たないので train / eval で挙動が同じ — 推論時のバグが減る。

LLaMA 以降の LLM は LayerNorm の bias 無し版 = RMSNorm (#08) を使うのが主流。NLP の系列モデルではほぼ全部 LayerNorm。

</details>

### Signature
```python
def my_layer_norm(
    x: torch.Tensor,      # input
    gamma: torch.Tensor,   # scale (最後の dim と同じ size)
    beta: torch.Tensor,    # shift (最後の dim と同じ size)
    eps: float = 1e-5
) -> torch.Tensor:
    ...
```

### Rules
- `F.layer_norm` や `torch.nn.LayerNorm` は **使わない**
- 最後の次元だけで normalize
- autograd をサポート

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q --force-reinstall --no-deps git+https://github.com/alextfkd/TorchCode.git')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def my_layer_norm(x, gamma, beta, eps=1e-5):
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False)
    x_norm = (x - mean) / torch.sqrt(var + eps)
    return gamma * x_norm + beta

In [ ]:
# Verify
x = torch.randn(2, 8)
gamma = torch.ones(8)
beta = torch.zeros(8)
out = my_layer_norm(x, gamma, beta)
ref = torch.nn.functional.layer_norm(x, [8], gamma, beta)
print("Match ref?", torch.allclose(out, ref, atol=1e-4))

In [ ]:
# Run judge
from torch_judge import check
check("layernorm")